# Attention Mechanisms: The Heart of Transformers

In this notebook, we'll build attention from scratch using only NumPy. By the end, you'll:

1. Understand **why** attention was invented
2. Implement **scaled dot-product attention** step by step
3. **Visualize** attention weights to see what the model "looks at"
4. Understand the difference between **self-attention** and **cross-attention**

**Prerequisites:** Basic Python, NumPy, and neural network fundamentals from `00-neural-networks`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# For nicer plots
matplotlib.rcParams['figure.figsize'] = (10, 6)
matplotlib.rcParams['font.size'] = 12

np.random.seed(42)  # For reproducibility
print("Setup complete!")

## Part 1: Why Do We Need Attention?

### The Problem with Older Models

Before transformers, models like RNNs processed words **one at a time**, left to right:

```
"The" → "cat" → "sat" → "on" → "the" → "mat" → "because" → "it" → "was" → "tired"
  ↓        ↓        ↓       ↓       ↓        ↓         ↓         ↓       ↓       ↓
 h₁  →   h₂   →   h₃  →  h₄  →  h₅   →   h₆   →   h₇   →   h₈  →  h₉  →  h₁₀
```

By the time the model reaches "it", the information about "cat" has been squeezed through many steps and may be lost.

### The Attention Solution

Attention lets **every word look at every other word directly** -- no matter how far apart they are:

```
"it" can directly look at:
  "The"     → low relevance  (0.03)
  "cat"     → HIGH relevance (0.70)  ← "it" refers to "cat"!
  "sat"     → low relevance  (0.02)
  ...       → ...
```

Let's build this step by step.

## Part 2: Word Embeddings (Words → Numbers)

Before we can do attention, we need to convert words into numbers. Each word becomes a **vector** (a list of numbers) called an **embedding**.

In real models, these embeddings are learned from data and have hundreds of dimensions. For learning, we'll use tiny 4-dimensional embeddings.

In [ ]:
# Our sentence: "The cat sat"
# In real models, embeddings come from a learned lookup table.
# Here we'll use small, hand-crafted ones so we can see what's happening.

words = ["The", "cat", "sat"]
d_model = 4  # Embedding dimension (real models use 512, 768, etc.)

# Each word is represented as a vector of d_model numbers
embeddings = {
    "The": np.array([1.0, 0.0, 0.0, 0.1]),   # Articles
    "cat": np.array([0.0, 1.0, 0.8, 0.0]),    # Nouns/animals
    "sat": np.array([0.0, 0.5, 0.0, 1.0]),    # Verbs/actions
}

# Stack into a matrix: each ROW is one word's embedding
X = np.array([embeddings[w] for w in words])

print("Input matrix X (each row = one word):")
print(f"Shape: {X.shape}  (3 words × 4 dimensions)")
print()
for i, word in enumerate(words):
    print(f"  '{word}': {X[i]}")

## Part 3: Creating Q, K, V (Query, Key, Value)

The core of attention uses three different "views" of each word:

- **Query (Q)**: What this word is *looking for*
- **Key (K)**: What this word *advertises* about itself
- **Value (V)**: The actual *information* this word carries

### The Library Analogy

Think of it like searching a library:
- **Q** = Your search question ("I want books about dinosaurs")
- **K** = Labels on each shelf ("Science", "History", "Fiction")
- **V** = The actual books on each shelf

You compare your question (Q) to each shelf label (K), and grab books (V) from the best-matching shelves.

### How Q, K, V Are Created

We multiply each word's embedding by three different **weight matrices** (W_Q, W_K, W_V):

In [ ]:
# In real transformers, these weight matrices are LEARNED during training.
# Here we initialize them randomly to demonstrate the concept.

d_k = 3  # Dimension of queries and keys (can differ from d_model)
d_v = 3  # Dimension of values

# Weight matrices (these would be learned in a real model)
W_Q = np.random.randn(d_model, d_k) * 0.5  # Shape: (4, 3)
W_K = np.random.randn(d_model, d_k) * 0.5  # Shape: (4, 3)
W_V = np.random.randn(d_model, d_v) * 0.5  # Shape: (4, 3)

print("Weight matrices (learned during training):")
print(f"  W_Q shape: {W_Q.shape}  (d_model × d_k = {d_model} × {d_k})")
print(f"  W_K shape: {W_K.shape}  (d_model × d_k = {d_model} × {d_k})")
print(f"  W_V shape: {W_V.shape}  (d_model × d_v = {d_model} × {d_v})")

In [ ]:
# Create Q, K, V by multiplying embeddings by weight matrices
Q = X @ W_Q  # (3 words × 4 dims) @ (4 dims × 3 d_k) = (3 words × 3 d_k)
K = X @ W_K  # Same shape
V = X @ W_V  # Same shape

print("Q (Queries - what each word is looking for):")
for i, word in enumerate(words):
    print(f"  Q_{word}: {np.round(Q[i], 3)}")

print(f"\nK (Keys - what each word advertises):")
for i, word in enumerate(words):
    print(f"  K_{word}: {np.round(K[i], 3)}")

print(f"\nV (Values - actual information each word carries):")
for i, word in enumerate(words):
    print(f"  V_{word}: {np.round(V[i], 3)}")

## Part 4: Computing Attention Scores (Q × K^T)

Now we measure how well each Query matches each Key using the **dot product**.

### What's a Dot Product?

The dot product measures how similar two vectors point in the same direction:
- **High positive** = very similar
- **Near zero** = unrelated
- **Negative** = opposite

```
Example:
  [1, 0, 1] · [1, 0, 1] = 1×1 + 0×0 + 1×1 = 2   (SAME direction → high!)
  [1, 0, 1] · [0, 1, 0] = 1×0 + 0×1 + 1×0 = 0   (perpendicular → zero!)
```

In [ ]:
# Compute attention scores: Q × K^T
# This gives us a score for every (query word, key word) pair

scores = Q @ K.T  # (3 words × 3 d_k) @ (3 d_k × 3 words) = (3 × 3)

print("Attention scores (Q × K^T):")
print(f"Shape: {scores.shape}  (every word scored against every other word)\n")

# Display as a nice table
print(f"{'':>8}", end="")
for word in words:
    print(f"{word:>10}", end="")
print("  ← Keys")
print(f"{'':>8}" + "-" * 30)
for i, word in enumerate(words):
    print(f"{word:>6} |", end="")
    for j in range(len(words)):
        print(f"{scores[i, j]:>10.3f}", end="")
    print()
print(f"{'':>8}↑ Queries")
print(f"\nEach cell = how much the row word's Query matches the column word's Key")

## Part 5: Scaling (÷ √d_k)

We divide the scores by √d_k to prevent them from getting too large.

### Why scale?

When vectors have many dimensions, dot products tend to produce large numbers. Large numbers cause softmax to become too "sharp" -- almost all weight goes to one word, and the model can't blend information from multiple sources.

In [ ]:
# Scale by sqrt(d_k)
scale = np.sqrt(d_k)
scaled_scores = scores / scale

print(f"Scaling factor: √d_k = √{d_k} = {scale:.3f}")
print(f"\nBefore scaling: scores range from {scores.min():.3f} to {scores.max():.3f}")
print(f"After scaling:  scores range from {scaled_scores.min():.3f} to {scaled_scores.max():.3f}")
print("\nScaled scores:")
print(np.round(scaled_scores, 3))

# Demonstrate why scaling matters
print("\n--- Why scaling matters ---")
large_scores = np.array([100.0, 1.0, 2.0])
small_scores = np.array([3.0, 1.0, 2.0])

def softmax(x):
    exp_x = np.exp(x - np.max(x))  # Subtract max for numerical stability
    return exp_x / exp_x.sum()

print(f"Unscaled scores {large_scores} → softmax: {np.round(softmax(large_scores), 6)}")
print(f"  → Almost ALL weight on first element! Can't blend information.")
print(f"Scaled scores   {small_scores} → softmax: {np.round(softmax(small_scores), 4)}")
print(f"  → Weights are spread out. Model can blend from multiple words.")

## Part 6: Softmax (Scores → Weights)

**Softmax** converts raw scores into probabilities:
- All values become positive
- They sum to exactly 1.0
- Bigger inputs → bigger outputs

Think of it as converting test scores into percentages.

In [ ]:
def softmax(x):
    """Compute softmax along the last axis."""
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))  # Numerical stability
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

# Apply softmax to each row (each word's attention distribution)
attention_weights = softmax(scaled_scores)

print("Attention weights (after softmax):")
print(f"Shape: {attention_weights.shape}\n")

print(f"{'':>8}", end="")
for word in words:
    print(f"{word:>10}", end="")
print(f"{'Sum':>10}")
print(f"{'':>8}" + "-" * 40)
for i, word in enumerate(words):
    print(f"{word:>6} |", end="")
    for j in range(len(words)):
        print(f"{attention_weights[i, j]:>10.4f}", end="")
    print(f"{attention_weights[i].sum():>10.4f}")

print("\nEach row sums to 1.0 — these are the attention weights!")
print("Higher values = more attention paid to that word.")

## Part 7: Visualize the Attention Weights

Let's create a heatmap to see what each word "pays attention to":

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

im = ax.imshow(attention_weights, cmap='Blues', aspect='auto')

# Labels
ax.set_xticks(range(len(words)))
ax.set_yticks(range(len(words)))
ax.set_xticklabels(words, fontsize=14)
ax.set_yticklabels(words, fontsize=14)
ax.set_xlabel('Keys (attending TO)', fontsize=13)
ax.set_ylabel('Queries (attending FROM)', fontsize=13)
ax.set_title('Attention Weights\n(darker = more attention)', fontsize=14)

# Add text annotations
for i in range(len(words)):
    for j in range(len(words)):
        color = 'white' if attention_weights[i, j] > 0.5 else 'black'
        ax.text(j, i, f'{attention_weights[i, j]:.3f}',
                ha='center', va='center', fontsize=14, color=color)

plt.colorbar(im, ax=ax, label='Attention Weight')
plt.tight_layout()
plt.show()

## Part 8: Weighted Sum of Values (The Final Output)

The last step: multiply the attention weights by the Value vectors to get a **weighted combination** of information.

Each output word is now a blend of all input words, weighted by how relevant they are.

In [ ]:
# Final output = attention_weights × V
output = attention_weights @ V  # (3 × 3) @ (3 × 3) = (3 × 3)

print("Output of attention (weighted sum of values):")
print(f"Shape: {output.shape}\n")

for i, word in enumerate(words):
    print(f"  output for '{word}': {np.round(output[i], 4)}")
    # Show the breakdown
    parts = []
    for j, w2 in enumerate(words):
        parts.append(f"{attention_weights[i, j]:.3f} × V_{w2}")
    print(f"    = {' + '.join(parts)}")
    print()

## Part 9: The Complete Attention Function

Let's put it all together into one clean function:

```
                        Q × K^T
Attention(Q, K, V) = softmax( ─────── ) × V
                         √d_k
```

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute Scaled Dot-Product Attention.
    
    Args:
        Q: Queries  (num_words × d_k)
        K: Keys     (num_words × d_k)
        V: Values   (num_words × d_v)
        mask: Optional mask to prevent attending to certain positions
    
    Returns:
        output: Weighted sum of values (num_words × d_v)
        weights: Attention weights     (num_words × num_words)
    """
    d_k = Q.shape[-1]
    
    # Step 1: Compute similarity scores
    scores = Q @ K.T  # (num_words × num_words)
    
    # Step 2: Scale
    scores = scores / np.sqrt(d_k)
    
    # Step 3: Apply mask (if provided)
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)
    
    # Step 4: Softmax to get weights
    weights = softmax(scores)
    
    # Step 5: Weighted sum of values
    output = weights @ V
    
    return output, weights

# Test it!
output, weights = scaled_dot_product_attention(Q, K, V)
print("Attention output shape:", output.shape)
print("Attention weights shape:", weights.shape)
print("\nAttention weights:")
print(np.round(weights, 4))

## Part 10: Causal (Decoder) Masking

In decoder models (like GPT), each word can only attend to words that came **before** it. We achieve this with a **causal mask**.

The mask sets "future" attention scores to negative infinity before softmax, making those weights effectively zero.

In [ ]:
# Create a causal mask (lower triangular matrix)
seq_len = len(words)
causal_mask = np.tril(np.ones((seq_len, seq_len)))  # Lower triangle of 1s

print("Causal mask (1 = can attend, 0 = cannot attend):")
print(f"{'':>8}", end="")
for word in words:
    print(f"{word:>8}", end="")
print()
for i, word in enumerate(words):
    print(f"{word:>6} |", end="")
    for j in range(len(words)):
        symbol = "  CAN" if causal_mask[i, j] == 1 else "    X"
        print(f"{symbol:>8}", end="")
    print()

print("\n'The' can only see itself")
print("'cat' can see 'The' and itself")
print("'sat' can see everything before it")

In [ ]:
# Apply causal masking
output_masked, weights_masked = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

# Visualize both side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, w, title in [
    (axes[0], weights, 'Self-Attention (Encoder)\nBidirectional: see all words'),
    (axes[1], weights_masked, 'Causal Attention (Decoder)\nLeft-to-right: only see past words')
]:
    im = ax.imshow(w, cmap='Blues', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(words)))
    ax.set_yticks(range(len(words)))
    ax.set_xticklabels(words, fontsize=13)
    ax.set_yticklabels(words, fontsize=13)
    ax.set_xlabel('Keys (attending TO)', fontsize=12)
    ax.set_ylabel('Queries (attending FROM)', fontsize=12)
    ax.set_title(title, fontsize=13)
    for i in range(len(words)):
        for j in range(len(words)):
            color = 'white' if w[i, j] > 0.5 else 'black'
            ax.text(j, i, f'{w[i, j]:.3f}',
                    ha='center', va='center', fontsize=13, color=color)

plt.tight_layout()
plt.show()
print("Notice: In causal attention, the upper-right triangle is zeroed out.")
print("Each word can only attend to itself and previous words.")

## Part 11: A Longer Example

Let's try a more realistic sentence to see attention in action:

In [ ]:
# Longer sentence with more interesting attention patterns
sentence = ["The", "cat", "sat", "on", "the", "mat"]
n_words = len(sentence)
d_model_big = 8
d_k_big = 6

# Random embeddings (in real models these are learned)
np.random.seed(123)
X_big = np.random.randn(n_words, d_model_big) * 0.5

# Make similar words have similar embeddings
# "The" and "the" should be similar
X_big[4] = X_big[0] + np.random.randn(d_model_big) * 0.1
# "cat" and "mat" rhyme and are both nouns
X_big[5] = X_big[1] * 0.3 + np.random.randn(d_model_big) * 0.3

# Random weight matrices
W_Q_big = np.random.randn(d_model_big, d_k_big) * 0.3
W_K_big = np.random.randn(d_model_big, d_k_big) * 0.3
W_V_big = np.random.randn(d_model_big, d_k_big) * 0.3

Q_big = X_big @ W_Q_big
K_big = X_big @ W_K_big
V_big = X_big @ W_V_big

output_big, weights_big = scaled_dot_product_attention(Q_big, K_big, V_big)

# Visualize
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(weights_big, cmap='Blues', aspect='auto')
ax.set_xticks(range(n_words))
ax.set_yticks(range(n_words))
ax.set_xticklabels(sentence, fontsize=13)
ax.set_yticklabels(sentence, fontsize=13)
ax.set_xlabel('Attending TO (Keys)', fontsize=13)
ax.set_ylabel('Attending FROM (Queries)', fontsize=13)
ax.set_title('Attention Weights: "The cat sat on the mat"', fontsize=14)

for i in range(n_words):
    for j in range(n_words):
        color = 'white' if weights_big[i, j] > 0.4 else 'black'
        ax.text(j, i, f'{weights_big[i, j]:.2f}',
                ha='center', va='center', fontsize=11, color=color)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Part 12: Self-Attention vs. Cross-Attention

There are two types of attention:

- **Self-attention**: Q, K, V all come from the **same** sequence (words attending to each other)
- **Cross-attention**: Q comes from one sequence, K and V from another (e.g., translation)

Let's implement both:

In [ ]:
# === Self-Attention ===
# All Q, K, V come from the same sentence
english = ["The", "cat", "sat"]
np.random.seed(42)
X_eng = np.random.randn(3, 4)

W_Q_sa = np.random.randn(4, 3) * 0.5
W_K_sa = np.random.randn(4, 3) * 0.5
W_V_sa = np.random.randn(4, 3) * 0.5

Q_self = X_eng @ W_Q_sa  # From English sentence
K_self = X_eng @ W_K_sa  # From English sentence (SAME!)
V_self = X_eng @ W_V_sa  # From English sentence (SAME!)

_, self_attn_weights = scaled_dot_product_attention(Q_self, K_self, V_self)

# === Cross-Attention ===
# Q from French (decoder), K and V from English (encoder)
french = ["Le", "chat"]
X_fr = np.random.randn(2, 4)

W_Q_ca = np.random.randn(4, 3) * 0.5
W_K_ca = np.random.randn(4, 3) * 0.5
W_V_ca = np.random.randn(4, 3) * 0.5

Q_cross = X_fr @ W_Q_ca   # From French (decoder asks questions)
K_cross = X_eng @ W_K_ca  # From English (encoder provides context)
V_cross = X_eng @ W_V_ca  # From English (encoder provides context)

_, cross_attn_weights = scaled_dot_product_attention(Q_cross, K_cross, V_cross)

# Visualize both
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Self-attention
im1 = axes[0].imshow(self_attn_weights, cmap='Blues', aspect='auto')
axes[0].set_xticks(range(3))
axes[0].set_yticks(range(3))
axes[0].set_xticklabels(english, fontsize=13)
axes[0].set_yticklabels(english, fontsize=13)
axes[0].set_title('Self-Attention\n(English → English)', fontsize=14)
axes[0].set_xlabel('Keys (same sentence)', fontsize=12)
axes[0].set_ylabel('Queries (same sentence)', fontsize=12)
for i in range(3):
    for j in range(3):
        color = 'white' if self_attn_weights[i, j] > 0.5 else 'black'
        axes[0].text(j, i, f'{self_attn_weights[i, j]:.3f}',
                     ha='center', va='center', fontsize=13, color=color)

# Cross-attention
im2 = axes[1].imshow(cross_attn_weights, cmap='Oranges', aspect='auto')
axes[1].set_xticks(range(3))
axes[1].set_yticks(range(2))
axes[1].set_xticklabels(english, fontsize=13)
axes[1].set_yticklabels(french, fontsize=13)
axes[1].set_title('Cross-Attention\n(French → English)', fontsize=14)
axes[1].set_xlabel('Keys (English encoder)', fontsize=12)
axes[1].set_ylabel('Queries (French decoder)', fontsize=12)
for i in range(2):
    for j in range(3):
        color = 'white' if cross_attn_weights[i, j] > 0.5 else 'black'
        axes[1].text(j, i, f'{cross_attn_weights[i, j]:.3f}',
                     ha='center', va='center', fontsize=13, color=color)

plt.tight_layout()
plt.show()

print("Self-attention: each English word attends to other English words.")
print("Cross-attention: each French word attends to English words for translation context.")

## Summary

In this notebook, we built attention from scratch! Here's what we covered:

| Step | What Happens | Code |
|------|-------------|------|
| 1. Embed | Words → vectors | `X = embeddings[words]` |
| 2. Project | Create Q, K, V | `Q = X @ W_Q` |
| 3. Score | Measure similarity | `scores = Q @ K.T` |
| 4. Scale | Prevent extreme values | `scores / √d_k` |
| 5. Softmax | Scores → weights | `weights = softmax(scores)` |
| 6. Combine | Weighted sum of values | `output = weights @ V` |

### The Complete Formula

```
Attention(Q, K, V) = softmax(Q × K^T / √d_k) × V
```

### Key Takeaways

1. **Attention** lets every word look at every other word
2. **Q, K, V** are different learned projections of the same embeddings
3. **Dot product** measures how well a Query matches a Key
4. **Scaling** prevents softmax from becoming too sharp
5. **Causal masking** prevents looking at future words (used in GPT-like models)
6. **Self-attention** = same-sequence, **Cross-attention** = across sequences

**Next:** [Multi-Head Attention](./02_multi_head_attention.ipynb) — running multiple attention heads in parallel.